# 🛰️ Sentinel Protocol — Autonomous Decision-Time-Budget Engine

**Project:** Sentinel Protocol  
**Purpose:** This notebook implements a real-time threat classification engine for a planetary rover AI operating under communication latency constraints with Earth. Because a round-trip signal to Earth can take anywhere from 4 to 24+ minutes, the rover must autonomously decide whether a detected hazard allows time to wait for ground control guidance, requires a safe holding action while Earth is notified, or demands immediate autonomous action.

## Decision Tiers

| Tier | Label | Meaning |
|------|-------|---------|
| 1 | 🟢 GREEN | Time-to-harm exceeds 2× round-trip comm delay — wait for Earth response |
| 2 | 🟡 YELLOW | Time-to-harm is between 1× and 2× round-trip delay — take safe holding action, notify Earth |
| 3 | 🔴 RED | Time-to-harm is less than or equal to 1× round-trip delay — act immediately, notify Earth after |

---

In [13]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Literal
import textwrap


# ---------------------------------------------------------------------------
# Tier enumeration
# ---------------------------------------------------------------------------

class DecisionTier(Enum):
    GREEN  = "GREEN"   # safe to wait for Earth
    YELLOW = "YELLOW"  # hold + notify Earth
    RED    = "RED"     # act now, notify later


# ---------------------------------------------------------------------------
# Threat dataclass
# ---------------------------------------------------------------------------

ThreatType = Literal[
    "cliff_edge",
    "dust_storm",
    "battery_critical",
    "rockfall",
    "comms_blackout",
]

@dataclass
class Threat:
    """Represents a detected hazard encountered by the rover.

    Attributes
    ----------
    threat_type       : One of the five recognised threat categories.
    time_to_harm_s    : Estimated seconds until the hazard causes irreversible
                        damage or mission loss if no action is taken.
    comm_delay_s      : One-way communication delay to Earth in seconds.
                        A full command round-trip therefore costs 2 × this value.
    """
    threat_type:    ThreatType
    time_to_harm_s: float
    comm_delay_s:   float

    # Derived convenience properties
    @property
    def round_trip_s(self) -> float:
        """Full round-trip comm delay (signal to Earth + command back)."""
        return self.comm_delay_s * 2

    @property
    def time_margin_ratio(self) -> float:
        """Ratio of time-to-harm to round-trip delay.  >2 → GREEN, 1-2 → YELLOW, ≤1 → RED."""
        if self.round_trip_s == 0:
            return float("inf")
        return self.time_to_harm_s / self.round_trip_s


# ---------------------------------------------------------------------------
# Threat-specific urgency multipliers
# ---------------------------------------------------------------------------
# Some threats are inherently more dynamic (fast-moving rockfall) or
# statistically under-estimated (cliff_edge sensor noise).  A multiplier
# < 1 applies a conservatism penalty, effectively shrinking the perceived
# time window and pushing ambiguous cases toward higher tiers.

THREAT_CONSERVATISM: dict[str, float] = {
    "cliff_edge":       0.80,   # sensor noise → be conservative
    "dust_storm":       0.90,   # storm intensity can escalate quickly
    "battery_critical": 0.95,   # discharge rate is fairly predictable
    "rockfall":         0.70,   # highly dynamic, worst-case bias
    "comms_blackout":   1.00,   # predictable orbital geometry
}


# ---------------------------------------------------------------------------
# Core classification function
# ---------------------------------------------------------------------------

def classify_threat(
    threat_type:    ThreatType,
    time_to_harm_s: float,
    comm_delay_s:   float,
) -> DecisionTier:
    """Classify a rover threat into a decision tier.

    Parameters
    ----------
    threat_type       : Category of the detected hazard.
    time_to_harm_s    : Estimated seconds to irreversible harm.
    comm_delay_s      : One-way comm latency to Earth in seconds.

    Returns
    -------
    DecisionTier
        GREEN  — time_to_harm > 2 × round_trip  (adjusted for conservatism)
        YELLOW — round_trip < time_to_harm ≤ 2 × round_trip
        RED    — time_to_harm ≤ round_trip
    """
    if threat_type not in THREAT_CONSERVATISM:
        raise ValueError(f"Unknown threat type: {threat_type!r}")

    conservatism = THREAT_CONSERVATISM[threat_type]
    threat = Threat(
        threat_type=threat_type,
        time_to_harm_s=time_to_harm_s * conservatism,   # adjusted window
        comm_delay_s=comm_delay_s,
    )

    ratio = threat.time_margin_ratio

    if ratio > 2.0:
        return DecisionTier.GREEN
    elif ratio > 1.0:
        return DecisionTier.YELLOW
    else:
        return DecisionTier.RED


print("✅ Threat dataclass, classify_threat(), and conservatism table loaded.")

✅ Threat dataclass, classify_threat(), and conservatism table loaded.


---
## Test Cases — All Five Threat Types

Values are chosen to reflect realistic Mars-mission scenarios.  
Mars one-way comm delay ≈ 4–24 min (240–1440 s); used 780 s (~13 min) as a representative mid-range value.

In [14]:
# ---------------------------------------------------------------------------
# Test suite
# ---------------------------------------------------------------------------

COMM_DELAY_S = 780   # ~13-minute one-way delay (realistic Mars mid-range)

test_cases = [
    # (label, threat_type, time_to_harm_s, comm_delay_s, expected_tier)
    (
        "Cliff edge — rover is 4 m from a precipice, moving at 0.02 m/s\n"
        "  → time to reach edge ≈ 200 s.  Far less than round-trip delay.",
        "cliff_edge",
        200,          # 200 s to reach the edge
        COMM_DELAY_S,
        DecisionTier.RED,
    ),
    (
        "Dust storm — approaching storm front detected 90 min away\n"
        "  → 5400 s before solar panels are critically obscured.",
        "dust_storm",
        5400,         # 90 minutes
        COMM_DELAY_S,
        DecisionTier.GREEN,
    ),
    (
        "Battery critical — charge at 8%, estimated 40 min to full shutdown\n"
        "  → 2400 s window; just barely within one round-trip.",
        "battery_critical",
        2400,         # 40 minutes
        COMM_DELAY_S,
        DecisionTier.YELLOW,
    ),
    (
        "Rockfall — seismic sensor detects imminent slope collapse 8 s away\n"
        "  → near-instant hazard, immediate evasion required.",
        "rockfall",
        8,            # 8 seconds — geological event
        COMM_DELAY_S,
        DecisionTier.RED,
    ),
    (
        "Comms blackout — relay satellite occultation in 35 min\n"
        "  → 2100 s before loss of uplink; plenty of time to queue messages.",
        "comms_blackout",
        2100,         # 35 minutes
        COMM_DELAY_S,
        DecisionTier.YELLOW,
    ),
]


# ---------------------------------------------------------------------------
# Run and display
# ---------------------------------------------------------------------------

TIER_ICON = {
    DecisionTier.GREEN:  "🟢",
    DecisionTier.YELLOW: "🟡",
    DecisionTier.RED:    "🔴",
}

TIER_ACTION = {
    DecisionTier.GREEN:  "Wait for Earth response.",
    DecisionTier.YELLOW: "Execute safe holding action; notify Earth immediately.",
    DecisionTier.RED:    "Act autonomously NOW; notify Earth after action.",
}

all_pass = True
print("=" * 70)
print(f"  SENTINEL PROTOCOL — Decision Engine Test Run")
print(f"  Comm delay (one-way): {COMM_DELAY_S} s  |  Round-trip: {COMM_DELAY_S*2} s")
print("=" * 70)

for i, (description, threat_type, tth, comm, expected) in enumerate(test_cases, 1):
    result  = classify_threat(threat_type, tth, comm)
    passed  = result == expected
    all_pass = all_pass and passed
    status  = "PASS ✓" if passed else f"FAIL ✗  (expected {expected.value})"

    # Compute ratio for display (pre-conservatism raw ratio)
    raw_ratio = tth / (comm * 2)
    adj_ratio = (tth * THREAT_CONSERVATISM[threat_type]) / (comm * 2)

    print(f"\nTest {i} — {threat_type.upper().replace('_', ' ')}")
    for line in description.strip().splitlines():
        print(f"  {line}")
    print(f"  Time-to-harm : {tth:,} s")
    print(f"  Raw ratio    : {raw_ratio:.3f}  (time_to_harm / round_trip)")
    print(f"  Adj ratio    : {adj_ratio:.3f}  (after {THREAT_CONSERVATISM[threat_type]:.0%} conservatism)")
    print(f"  Decision     : {TIER_ICON[result]} {result.value}  →  {TIER_ACTION[result]}")
    print(f"  [{status}]")

print("\n" + "=" * 70)
if all_pass:
    print("  ✅ ALL TESTS PASSED — Decision engine nominal.")
else:
    print("  ❌ ONE OR MORE TESTS FAILED — Review classification logic.")
print("=" * 70)

  SENTINEL PROTOCOL — Decision Engine Test Run
  Comm delay (one-way): 780 s  |  Round-trip: 1560 s

Test 1 — CLIFF EDGE
  Cliff edge — rover is 4 m from a precipice, moving at 0.02 m/s
    → time to reach edge ≈ 200 s.  Far less than round-trip delay.
  Time-to-harm : 200 s
  Raw ratio    : 0.128  (time_to_harm / round_trip)
  Adj ratio    : 0.103  (after 80% conservatism)
  Decision     : 🔴 RED  →  Act autonomously NOW; notify Earth after action.
  [PASS ✓]

Test 2 — DUST STORM
  Dust storm — approaching storm front detected 90 min away
    → 5400 s before solar panels are critically obscured.
  Time-to-harm : 5,400 s
  Raw ratio    : 3.462  (time_to_harm / round_trip)
  Adj ratio    : 3.115  (after 90% conservatism)
  Decision     : 🟢 GREEN  →  Wait for Earth response.
  [PASS ✓]

Test 3 — BATTERY CRITICAL
  Battery critical — charge at 8%, estimated 40 min to full shutdown
    → 2400 s window; just barely within one round-trip.
  Time-to-harm : 2,400 s
  Raw ratio    : 1.538  (time

---
## Scenario Simulator — Tick-by-Tick Sensor Escalation

Each threat type has a **physics model** that drives one or more sensor readings forward each tick.  
The raw sensor state is converted to a `time_to_harm_s` estimate, which is fed directly into  
`classify_threat()` so the decision tier updates live as the scenario progresses.

| Threat | Sensor(s) modelled | Escalation mechanism |
|---|---|---|
| `cliff_edge` | `distance_m` (LiDAR) | Rover drifts toward edge each tick |
| `dust_storm` | `wind_speed_ms`, `dust_density_gcm3` | Both ramp up; combined opacity drives shutdown ETA |
| `battery_critical` | `charge_pct` | Discharge accelerates as systems draw more power |
| `rockfall` | `seismic_g`, `debris_distance_m` | Ground vibration rises; debris closes in fast |
| `comms_blackout` | `relay_elevation_deg` | Satellite arc descends toward horizon |

### `run_scenario(threat_type, ticks=20, comm_delay_s=780)`
A generator that yields a `TickState` named-tuple each tick containing:
- `tick` — tick index (0-based)
- `sensors` — `dict` of raw sensor readings for this threat type
- `time_to_harm_s` — derived estimate fed into the classifier
- `tier` — the `DecisionTier` result at this tick

In [15]:
from typing import Iterator, NamedTuple
import math


# ---------------------------------------------------------------------------
# Tick state — what the generator yields each tick
# ---------------------------------------------------------------------------

class TickState(NamedTuple):
    tick:           int
    sensors:        dict        # raw physics readings
    time_to_harm_s: float       # derived estimate → classify_threat input
    tier:           DecisionTier


# ---------------------------------------------------------------------------
# Per-threat physics models
# Each model is a generator that yields (sensors_dict, time_to_harm_s) per tick.
# ---------------------------------------------------------------------------

def _cliff_edge_model(ticks: int):
    """Rover drifts toward a precipice; wheel-slip accelerates closure rate."""
    distance_m   = 100.0   # initial LiDAR reading to cliff edge (metres)
    speed_ms     = 0.02    # initial drift speed m/s
    accel        = 0.003   # speed increase per tick (slip worsens)
    tick_dur_s   = 30      # real seconds represented by one tick
    for _ in range(ticks):
        time_to_harm = distance_m / speed_ms if speed_ms > 0 else float('inf')
        yield ({'distance_m': round(distance_m, 3),
                'drift_speed_ms': round(speed_ms, 4)},
               time_to_harm)
        distance_m = max(0.0, distance_m - speed_ms * tick_dur_s)
        speed_ms  += accel


def _dust_storm_model(ticks: int):
    """Wind and dust density rise; combined optical depth predicts panel shutdown."""
    wind_ms      = 4.0     # m/s — gentle breeze
    dust_gcm3    = 0.001   # g/cm³ — background haze
    wind_ramp    = 2.5     # m/s added per tick
    dust_ramp    = 0.004   # density increase per tick
    # Shutdown threshold: panels fail when optical_depth > 1.0
    # optical_depth proxy = dust_gcm3 * 100 * wind_ms^0.3
    # time_to_harm = seconds until optical_depth reaches 1.0, linear extrapolation
    tick_dur_s   = 60
    for _ in range(ticks):
        optical_depth = dust_gcm3 * 100 * (wind_ms ** 0.3)
        # rate of optical_depth change per second
        next_dust  = dust_gcm3 + dust_ramp
        next_wind  = wind_ms  + wind_ramp
        next_od    = next_dust * 100 * (next_wind ** 0.3)
        od_rate_per_s = max((next_od - optical_depth) / tick_dur_s, 1e-9)
        remaining_od  = max(1.0 - optical_depth, 0.0)
        time_to_harm  = remaining_od / od_rate_per_s
        yield ({'wind_speed_ms':     round(wind_ms, 2),
                'dust_density_gcm3': round(dust_gcm3, 4),
                'optical_depth':     round(optical_depth, 4)},
               max(time_to_harm, 0.0))
        wind_ms   = next_wind
        dust_gcm3 = next_dust


def _battery_critical_model(ticks: int):
    """Battery charge drains; draw rate accelerates as thermal systems kick in."""
    charge_pct   = 18.0    # %
    draw_pct_per_tick = 0.8  # base drain per tick
    draw_accel   = 0.12    # drain increase per tick (heaters, comms fighting)
    tick_dur_s   = 60
    for _ in range(ticks):
        # time to reach 0 % at current draw rate
        time_to_harm = (charge_pct / draw_pct_per_tick) * tick_dur_s
        yield ({'charge_pct':      round(charge_pct, 2),
                'draw_pct_per_tick': round(draw_pct_per_tick, 3)},
               time_to_harm)
        charge_pct      = max(0.0, charge_pct - draw_pct_per_tick)
        draw_pct_per_tick += draw_accel


def _rockfall_model(ticks: int):
    """Seismic intensity rises; debris front closes rapidly."""
    seismic_g     = 0.05   # gravitational acceleration units
    debris_dist_m = 80.0   # metres to the debris front
    debris_speed  = 1.5    # m/s — slow initial roll
    seismic_ramp  = 0.08
    speed_ramp    = 2.0    # debris accelerates under gravity
    tick_dur_s    = 5
    for _ in range(ticks):
        time_to_harm = debris_dist_m / debris_speed if debris_speed > 0 else float('inf')
        yield ({'seismic_g':      round(seismic_g, 3),
                'debris_dist_m':  round(debris_dist_m, 2),
                'debris_speed_ms': round(debris_speed, 2)},
               time_to_harm)
        debris_dist_m = max(0.0, debris_dist_m - debris_speed * tick_dur_s)
        debris_speed += speed_ramp
        seismic_g    += seismic_ramp


def _comms_blackout_model(ticks: int):
    """Relay satellite arc descends; time to LOS shrinks linearly then faster near horizon."""
    elevation_deg = 42.0   # degrees above horizon
    descent_rate  = 1.8    # deg per tick
    tick_dur_s    = 60
    for _ in range(ticks):
        # Time to reach 5-deg minimum elevation (operational cutoff)
        remaining_deg = max(elevation_deg - 5.0, 0.0)
        # descent accelerates slightly due to orbital geometry near horizon
        effective_rate = descent_rate * (1 + 0.04 * (42.0 - elevation_deg))
        time_to_harm   = (remaining_deg / effective_rate) * tick_dur_s
        yield ({'relay_elevation_deg': round(elevation_deg, 2),
                'effective_descent_rate': round(effective_rate, 3)},
               max(time_to_harm, 0.0))
        elevation_deg = max(0.0, elevation_deg - descent_rate)


_MODELS = {
    'cliff_edge':       _cliff_edge_model,
    'dust_storm':       _dust_storm_model,
    'battery_critical': _battery_critical_model,
    'rockfall':         _rockfall_model,
    'comms_blackout':   _comms_blackout_model,
}


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def run_scenario(
    threat_type:  ThreatType,
    ticks:        int   = 20,
    comm_delay_s: float = 780,
) -> Iterator[TickState]:
    """Tick-by-tick scenario simulator for a single threat type.

    Yields
    ------
    TickState
        tick           : Tick index (0-based).
        sensors        : Raw sensor readings dict for the current tick.
        time_to_harm_s : Derived time-to-harm estimate (seconds).
        tier           : DecisionTier result from classify_threat().
    """
    if threat_type not in _MODELS:
        raise ValueError(f'Unknown threat type: {threat_type!r}')
    model = _MODELS[threat_type](ticks)
    for tick_idx, (sensors, tth) in enumerate(model):
        tier = classify_threat(threat_type, tth, comm_delay_s)
        yield TickState(tick=tick_idx, sensors=sensors,
                        time_to_harm_s=round(tth, 1), tier=tier)


print('Scenario simulator loaded. Models: ' + ', '.join(_MODELS))

Scenario simulator loaded. Models: cliff_edge, dust_storm, battery_critical, rockfall, comms_blackout


---
### Demo Run — `cliff_edge` scenario (20 ticks)

The rover starts **12 m** from the edge drifting at **0.02 m/s**.  
Wheel-slip gradually increases the closure rate each tick until the decision escalates  
from 🟢 GREEN → 🟡 YELLOW → 🔴 RED.

In [16]:
TIER_ICON = {
    DecisionTier.GREEN:  'GREEN ',
    DecisionTier.YELLOW: 'YELLOW',
    DecisionTier.RED:    'RED   ',
}
TIER_BADGE = {
    DecisionTier.GREEN:  '[GREEN ] -->  Wait for Earth response.',
    DecisionTier.YELLOW: '[YELLOW] -->  Safe hold + notify Earth immediately.',
    DecisionTier.RED:    '[RED   ] -->  ACT NOW — notify Earth after.',
}

COMM_DELAY_S = 780

print('=' * 72)
print('  SENTINEL PROTOCOL — Scenario Simulator')
print('  Scenario : cliff_edge')
print(f'  Comm delay (one-way): {COMM_DELAY_S} s  |  Round-trip: {COMM_DELAY_S*2} s')
print('  Tick duration: 30 s real-world seconds per tick')
print('=' * 72)
print(f'  {"Tick":>4}  {"Dist(m)":>8}  {"Speed(m/s)":>10}  {"TTH(s)":>9}  {"Tier & Action"}')
print('-' * 72)

prev_tier = None
for state in run_scenario('cliff_edge', ticks=20, comm_delay_s=COMM_DELAY_S):
    d   = state.sensors['distance_m']
    spd = state.sensors['drift_speed_ms']
    tth = state.time_to_harm_s
    tier = state.tier

    # Mark tier transitions
    transition = '  <-- TIER CHANGE' if tier != prev_tier and prev_tier is not None else ''
    prev_tier = tier

    print(f'  {state.tick:>4}  {d:>8.3f}  {spd:>10.4f}  {tth:>9.1f}  {TIER_BADGE[tier]}{transition}')

print('=' * 72)
print('  Simulation complete.')
print('=' * 72)

  SENTINEL PROTOCOL — Scenario Simulator
  Scenario : cliff_edge
  Comm delay (one-way): 780 s  |  Round-trip: 1560 s
  Tick duration: 30 s real-world seconds per tick
  Tick   Dist(m)  Speed(m/s)     TTH(s)  Tier & Action
------------------------------------------------------------------------
     0   100.000      0.0200     5000.0  [GREEN ] -->  Wait for Earth response.
     1    99.400      0.0230     4321.7  [GREEN ] -->  Wait for Earth response.
     2    98.710      0.0260     3796.5  [YELLOW] -->  Safe hold + notify Earth immediately.  <-- TIER CHANGE
     3    97.930      0.0290     3376.9  [YELLOW] -->  Safe hold + notify Earth immediately.
     4    97.060      0.0320     3033.1  [YELLOW] -->  Safe hold + notify Earth immediately.
     5    96.100      0.0350     2745.7  [YELLOW] -->  Safe hold + notify Earth immediately.
     6    95.050      0.0380     2501.3  [YELLOW] -->  Safe hold + notify Earth immediately.
     7    93.910      0.0410     2290.5  [YELLOW] -->  Safe ho

---
## AI Reasoning Layer — IBM watsonx.ai (Granite)

Each tick's structured data is sent to a **watsonx.ai foundation model** (`ibm/granite-4-h-small`,
Frankfurt endpoint) which generates a single professional mission-log sentence — the kind a flight
engineer would write to justify an autonomous decision.

### Credentials
Loaded from a local `.env` file (never committed to git):
```
WATSONX_API_KEY=<your IBM Cloud API key>
WATSONX_PROJECT_ID=<your watsonx.ai project GUID>
```

### `generate_reasoning(tick_data)` — input schema
```python
{
    'threat_type':    str,   # e.g. 'cliff_edge'
    'sensors':        dict,  # raw sensor readings at this tick
    'time_to_harm_s': float, # seconds to irreversible harm
    'round_trip_s':   float, # full comm round-trip delay in seconds
    'ratio':          float, # adjusted TTH / round-trip ratio
    'tier':           str,   # 'GREEN' | 'YELLOW' | 'RED'
    'action':         str,   # prescribed action string
}
```

In [17]:
import os
from dotenv import load_dotenv
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

# ---------------------------------------------------------------------------
# Load credentials from .env
# ---------------------------------------------------------------------------
load_dotenv()  # reads .env from the current working directory

WATSONX_API_KEY    = os.environ['WATSONX_API_KEY']
WATSONX_PROJECT_ID = os.environ['WATSONX_PROJECT_ID']
WATSONX_URL        = 'https://eu-de.ml.cloud.ibm.com'
WATSONX_MODEL_ID   = 'ibm/granite-4-h-small'

# ---------------------------------------------------------------------------
# Instantiate the model client
# granite-4-h-small is a chat model — use the /ml/v1/text/chat endpoint
# via .chat(), NOT .generate_text() which targets the deprecated completion API.
# ---------------------------------------------------------------------------
_wx_model = ModelInference(
    model_id    = WATSONX_MODEL_ID,
    credentials = Credentials(
        url     = WATSONX_URL,
        api_key = WATSONX_API_KEY,
    ),
    project_id  = WATSONX_PROJECT_ID,
)

# Chat parameters passed per-call (not stored on the client for chat API)
_CHAT_PARAMS = {
    'max_tokens':        80,
    'temperature':       0.3,
    'repetition_penalty': 1.05,
}


# ---------------------------------------------------------------------------
# System prompt (sets the persona once, reused across calls)
# ---------------------------------------------------------------------------
_SYSTEM_PROMPT = (
    'You are the autonomous reasoning system of a planetary rover named Sentinel. '
    'Write a single professional mission-log sentence (maximum 40 words) explaining '
    'the decision made. Be factual, precise, and terse — like a flight engineer '
    'writing a flight log entry. Output only the log sentence, nothing else.'
)


# ---------------------------------------------------------------------------
# User message template (the structured situation fed per tick)
# ---------------------------------------------------------------------------
_USER_TEMPLATE = (
    'SITUATION:\n'
    '  Threat      : {threat_type}\n'
    '  Sensor data : {sensors}\n'
    '  Time-to-harm: {time_to_harm_s:.1f} s\n'
    '  Round-trip comm delay: {round_trip_s:.0f} s\n'
    '  Adjusted ratio (TTH/RTT): {ratio:.3f}\n'
    '  Decision tier: {tier}\n'
    '  Required action: {action}\n'
    '\n'
    'Write the mission log entry for this tick.'
)


# ---------------------------------------------------------------------------
# Public function
# ---------------------------------------------------------------------------

def generate_reasoning(tick_data: dict) -> str:
    """Call watsonx.ai to generate a mission-log sentence for one tick.

    Uses the chat API (/ml/v1/text/chat) which is the correct endpoint for
    ibm/granite-4-h-small and avoids the deprecated text/generation endpoint.

    Parameters
    ----------
    tick_data : dict with keys:
        threat_type, sensors, time_to_harm_s, round_trip_s, ratio, tier, action

    Returns
    -------
    str  — a single mission-log sentence generated by the foundation model.
    """
    messages = [
        {'role': 'system', 'content': _SYSTEM_PROMPT},
        {'role': 'user',   'content': _USER_TEMPLATE.format(**tick_data)},
    ]
    response = _wx_model.chat(messages=messages, params=_CHAT_PARAMS)
    return response['choices'][0]['message']['content'].strip()


print(f'watsonx chat client ready  model={WATSONX_MODEL_ID}  url={WATSONX_URL}')

watsonx chat client ready  model=ibm/granite-4-h-small  url=https://eu-de.ml.cloud.ibm.com


---
### Live Demo — `generate_reasoning()` at tick 9 (GREEN → RED transition)

Tick 9 is the first **RED** tick in the `cliff_edge` simulation: distance has closed to
**91.36 m**, drift speed has accelerated to **0.047 m/s**, time-to-harm is **1943.8 s** —
below the 1560 s round-trip threshold. The AI layer explains *why* the rover must act now.

In [18]:
# ---------------------------------------------------------------------------
# Reconstruct tick-9 state from the cliff_edge scenario
# ---------------------------------------------------------------------------
COMM_DELAY_S = 780

tick9_state = None
for state in run_scenario('cliff_edge', ticks=20, comm_delay_s=COMM_DELAY_S):
    if state.tick == 9:
        tick9_state = state
        break

# Build the tick_data dict that generate_reasoning() expects
conservatism  = THREAT_CONSERVATISM['cliff_edge']
round_trip_s  = COMM_DELAY_S * 2
adj_tth       = tick9_state.time_to_harm_s * conservatism
ratio         = adj_tth / round_trip_s

TIER_ACTION_MAP = {
    DecisionTier.GREEN:  'Wait for Earth response.',
    DecisionTier.YELLOW: 'Execute safe holding action; notify Earth immediately.',
    DecisionTier.RED:    'Act autonomously NOW; notify Earth after action.',
}

tick_data = {
    'threat_type':    'cliff_edge',
    'sensors':        tick9_state.sensors,
    'time_to_harm_s': tick9_state.time_to_harm_s,
    'round_trip_s':   round_trip_s,
    'ratio':          ratio,
    'tier':           tick9_state.tier.value,
    'action':         TIER_ACTION_MAP[tick9_state.tier],
}

# ---------------------------------------------------------------------------
# Print tick summary
# ---------------------------------------------------------------------------
print('=' * 68)
print('  SENTINEL PROTOCOL — AI Reasoning Layer Demo')
print('  Tick 9 | cliff_edge | TIER TRANSITION: YELLOW -> RED')
print('=' * 68)
print(f"  Threat      : {tick_data['threat_type']}")
print(f"  Sensors     : {tick_data['sensors']}")
print(f"  Time-to-harm: {tick_data['time_to_harm_s']} s")
print(f"  Round-trip  : {tick_data['round_trip_s']} s")
print(f"  Adj ratio   : {tick_data['ratio']:.3f}")
print(f"  Tier        : {tick_data['tier']}")
print(f"  Action      : {tick_data['action']}")
print('-' * 68)
print('  Querying watsonx.ai...')

# ---------------------------------------------------------------------------
# Call the model
# ---------------------------------------------------------------------------
log_entry = generate_reasoning(tick_data)

print()
print('  MISSION LOG ENTRY (generated by ibm/granite-4-h-small):')
print(f'  "{log_entry}"')
print('=' * 68)

  SENTINEL PROTOCOL — AI Reasoning Layer Demo
  Tick 9 | cliff_edge | TIER TRANSITION: YELLOW -> RED
  Threat      : cliff_edge
  Sensors     : {'distance_m': 91.36, 'drift_speed_ms': 0.047}
  Time-to-harm: 1943.8 s
  Round-trip  : 1560 s
  Adj ratio   : 0.997
  Tier        : RED
  Action      : Act autonomously NOW; notify Earth after action.
--------------------------------------------------------------------
  Querying watsonx.ai...

  MISSION LOG ENTRY (generated by ibm/granite-4-h-small):
  "Sentinel autonomously executed evasive maneuvers to avoid cliff edge, initiating immediate hazard mitigation protocol per RED-tier directive; Earth notification scheduled post-action completion."
